# Embedding worker — Google Colab
Tạo `embedding_bundle.zip` bằng `python collab/build_bundles.py` trên máy trước, rồi chạy các ô theo thứ tự. FastAPI vẫn chạy trên máy và phải có URL HTTPS Colab truy cập được.

In [1]:
%pip install --force-reinstall --no-cache-dir Pillow==12.3.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 28.8 MB/s eta 0:00:00
  Attempting uninstall: Pillow
    Found existing installation: pillow 11.3.0
    Uninstalling pillow-11.3.0:
      Successfully uninstalled pillow-11.3.0


In [1]:
from google.colab import files
from pathlib import Path
from zipfile import ZipFile
import io, os, sys, subprocess


In [2]:

expected = 'embedding_bundle.zip'
uploaded = files.upload()
if expected not in uploaded:
    raise ValueError(f'Cần tải lên {expected}')
runtime = Path('/content/rag_colab_embedding')
runtime.mkdir(parents=True, exist_ok=True)
with ZipFile(io.BytesIO(uploaded[expected])) as archive:
    for member in archive.namelist():
        path = Path(member)
        if not member.startswith('gen/') or path.is_absolute() or '..' in path.parts or '\\' in member:
            raise ValueError(f'Đường dẫn không hợp lệ trong gói: {member}')
    archive.extractall(runtime)
project = runtime / 'gen'
os.chdir(project)
sys.path.insert(0, str(project))
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(project / 'requirements-colab.txt')])
print('Đã chuẩn bị mã embedding tại', project)

Saving embedding_bundle.zip to embedding_bundle.zip
Đã chuẩn bị mã embedding tại /content/rag_colab_embedding/gen


In [3]:
from getpass import getpass
from urllib.request import urlopen

os.environ['API_BASE_URL'] = input('FastAPI HTTPS URL: ').strip().rstrip('/')
os.environ['WORKER_TOKEN'] = getpass('WORKER_TOKEN: ')
os.environ['SUPABASE_URL'] = input('SUPABASE_URL: ').strip()
os.environ['SUPABASE_KEY'] = getpass('SUPABASE_KEY: ')
os.environ['SUPABASE_BUCKET'] = input('SUPABASE_BUCKET [rag-data]: ').strip() or 'rag-data'
if not os.environ['API_BASE_URL'].startswith('https://'):
    raise ValueError('Colab cần URL FastAPI HTTPS công khai')
with urlopen(os.environ['API_BASE_URL'] + '/health', timeout=15) as response:
    print('FastAPI health:', response.status, response.read().decode()[:200])

FastAPI HTTPS URL: https://terminals-mozilla-employer-retain.trycloudflare.com
WORKER_TOKEN: ··········
SUPABASE_URL: https://yrmkrkcnmuqedqboarpe.supabase.co
SUPABASE_KEY: ··········
SUPABASE_BUCKET [rag-data]: rag-data
FastAPI health: 200 {"status":"ok"}


In [4]:
import os
import sys
from pathlib import Path

matches = list(Path("/content").rglob("config/settings.py"))
print("Tìm thấy:", matches)

if not matches:
    raise FileNotFoundError("Không tìm thấy bundle đã giải nén")

project = matches[0].parents[1]
os.chdir(project)
sys.path.insert(0, str(project))

print("Project:", project)

Tìm thấy: [PosixPath('/content/rag_colab_embedding/gen/config/settings.py')]
Project: /content/rag_colab_embedding/gen


In [5]:
from config.settings import get_settings
get_settings.cache_clear()

from workers.colab.embedding_worker import start_embedding_worker
start_embedding_worker(worker_id="colab-embedding-1")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  813MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/408 [00:00<?, ?it/s]

Worker poll nhận lỗi tạm thời 530; sẽ tự thử lại
Worker poll nhận lỗi tạm thời 530; sẽ tự thử lại
Worker poll nhận lỗi tạm thời 530; sẽ tự thử lại
Worker poll nhận lỗi tạm thời 530; sẽ tự thử lại
Worker poll nhận lỗi tạm thời 530; sẽ tự thử lại
Worker poll nhận lỗi tạm thời 530; sẽ tự thử lại
Worker poll nhận lỗi tạm thời 530; sẽ tự thử lại
Worker poll nhận lỗi tạm thời 530; sẽ tự thử lại
Worker poll nhận lỗi tạm thời 530; sẽ tự thử lại
Worker poll nhận lỗi tạm thời 530; sẽ tự thử lại
Worker poll nhận lỗi tạm thời 530; sẽ tự thử lại
Worker poll nhận lỗi tạm thời 530; sẽ tự thử lại
Worker poll nhận lỗi tạm thời 530; sẽ tự thử lại
Worker poll nhận lỗi tạm thời 530; sẽ tự thử lại
Worker poll nhận lỗi tạm thời 530; sẽ tự thử lại
Worker poll nhận lỗi tạm thời 530; sẽ tự thử lại
Worker poll nhận lỗi tạm thời 530; sẽ tự thử lại
Worker poll nhận lỗi tạm thời 530; sẽ tự thử lại
Worker poll nhận lỗi tạm thời 530; sẽ tự thử lại
Worker poll nhận lỗi tạm thời 530; sẽ tự thử lại
Worker poll nhận lỗi

KeyboardInterrupt: 